In [1]:
from pyspark.sql.functions import col
from utils.spark_manager import SparkManager
from utils.data_processor import DataProcessor
from configs.path import HDFS_RAW_PATH

spark_manager = SparkManager(app_name="DataProcessingDemo")
spark = spark_manager.get_session()
processor = DataProcessor()

print(f" Loading raw data from {HDFS_RAW_PATH}...")
raw_df = spark.read.parquet(HDFS_RAW_PATH)

print(" Raw data loaded successfully. Showing a sample:")
raw_df.show(5, truncate=False)

 Initializing Spark session for 'DataProcessingDemo'...


25/08/29 20:32:35 WARN Utils: Your hostname, LAPTOP-E902JBKI.localdomain resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/08/29 20:32:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/29 20:32:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/08/29 20:32:37 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/08/29 20:32:37 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Spark session created successfully.
 Loading raw data from hdfs://localhost:9000/user/air_quality/raw...


 Raw data loaded successfully. Showing a sample:
+----------+--------+----+--------+------+------+----------+-----+---------+-----+---------+--------+----+----+------+
|Date      |Time    |COGT|PT08S1CO|NMHCGT|C6H6GT|PT08S2NMHC|NOxGT|PT08S3NOx|NO2GT|PT08S4NO2|PT08S5O3|T   |RH  |AH    |
+----------+--------+----+--------+------+------+----------+-----+---------+-----+---------+--------+----+----+------+
|10/03/2004|18.00.00|2.6 |1360.0  |150.0 |11.9  |1046.0    |166.0|1056.0   |113.0|1692.0   |1268.0  |13.6|48.9|0.7578|
|10/03/2004|19.00.00|2.0 |1292.0  |112.0 |9.4   |955.0     |103.0|1174.0   |92.0 |1559.0   |972.0   |13.3|47.7|0.7255|
|10/03/2004|20.00.00|2.2 |1402.0  |88.0  |9.0   |939.0     |131.0|1140.0   |114.0|1555.0   |1074.0  |11.9|54.0|0.7502|
|10/03/2004|21.00.00|2.2 |1376.0  |80.0  |9.2   |948.0     |172.0|1092.0   |122.0|1584.0   |1203.0  |11.0|60.0|0.7867|
|10/03/2004|22.00.00|1.6 |1272.0  |51.0  |6.5   |836.0     |131.0|1205.0   |116.0|1490.0   |1110.0  |11.2|59.6|0.7888|

In [2]:
df_step1 = processor._initial_cleaning(raw_df)

print("Step 1 Complete: Dropped duplicates and irrelevant sensor columns.")
print("DataFrame schema after cleaning:")
df_step1.printSchema()
df_step1.show(5, truncate=False)

- Step 1: Dropping duplicates and irrelevant columns...
Step 1 Complete: Dropped duplicates and irrelevant sensor columns.
DataFrame schema after cleaning:
root
 |-- Date: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- COGT: double (nullable = true)
 |-- NMHCGT: double (nullable = true)
 |-- C6H6GT: double (nullable = true)
 |-- NOxGT: double (nullable = true)
 |-- NO2GT: double (nullable = true)

+----------+--------+------+------+------+-----+-----+
|Date      |Time    |COGT  |NMHCGT|C6H6GT|NOxGT|NO2GT|
+----------+--------+------+------+------+-----+-----+
|12/03/2004|04.00.00|-200.0|10.0  |1.1   |21.0 |32.0 |
|12/03/2004|12.00.00|2.1   |114.0 |10.2  |143.0|113.0|
|12/03/2004|15.00.00|2.9   |185.0 |14.2  |190.0|126.0|
|11/03/2004|10.00.00|1.7   |77.0  |6.3   |112.0|98.0 |
|11/03/2004|02.00.00|0.9   |24.0  |2.3   |45.0 |60.0 |
+----------+--------+------+------+------+-----+-----+
only showing top 5 rows



In [3]:
df_step2 = processor._rename_columns(df_step1)

print("Step 2 Complete: Renamed columns for clarity.")
print("DataFrame schema after renaming:")
df_step2.printSchema()
df_step2.show(5, truncate=False)

- Step 2: Renaming columns...
Step 2 Complete: Renamed columns for clarity.
DataFrame schema after renaming:
root
 |-- date: string (nullable = true)
 |-- time: string (nullable = true)
 |-- carbon_monoxide: double (nullable = true)
 |-- non_methane_hydrocarbon: double (nullable = true)
 |-- benzene: double (nullable = true)
 |-- nitrogen_oxides: double (nullable = true)
 |-- nitrogen_dioxide: double (nullable = true)

+----------+--------+---------------+-----------------------+-------+---------------+----------------+
|date      |time    |carbon_monoxide|non_methane_hydrocarbon|benzene|nitrogen_oxides|nitrogen_dioxide|
+----------+--------+---------------+-----------------------+-------+---------------+----------------+
|12/03/2004|04.00.00|-200.0         |10.0                   |1.1    |21.0           |32.0            |
|12/03/2004|12.00.00|2.1            |114.0                  |10.2   |143.0          |113.0           |
|12/03/2004|15.00.00|2.9            |185.0                  |1

In [4]:
df_step3 = processor._handle_missing_values(df_step2)

print("Step 3 Complete: Replaced placeholder -200 values with null.")
print("Showing a sample of rows where carbon_monoxide was null:")
df_step3.filter(col("carbon_monoxide").isNull()).show(5, truncate=False)

- Step 3: Replacing -200 with null...
Step 3 Complete: Replaced placeholder -200 values with null.
Showing a sample of rows where carbon_monoxide was null:
+----------+--------+---------------+-----------------------+-------+---------------+----------------+
|date      |time    |carbon_monoxide|non_methane_hydrocarbon|benzene|nitrogen_oxides|nitrogen_dioxide|
+----------+--------+---------------+-----------------------+-------+---------------+----------------+
|12/03/2004|04.00.00|NULL           |10.0                   |1.1    |21.0           |32.0            |
|12/03/2004|09.00.00|NULL           |NULL                   |22.1   |NULL           |NULL            |
|11/03/2004|04.00.00|NULL           |14.0                   |1.3    |21.0           |34.0            |
+----------+--------+---------------+-----------------------+-------+---------------+----------------+



In [5]:
df_step4 = processor._impute_missing_values(df_step3)

print("Step 4 Complete: Imputed null values with the column average.")
print("Showing the same sample, now with imputed values:")
df_step4.filter(col("carbon_monoxide").isNull()).show(5, truncate=False)
df_step4.show(5, truncate=False)

- Step 4: Imputing nulls with column averages...
Step 4 Complete: Imputed null values with the column average.
Showing the same sample, now with imputed values:
+----+----+---------------+-----------------------+-------+---------------+----------------+
|date|time|carbon_monoxide|non_methane_hydrocarbon|benzene|nitrogen_oxides|nitrogen_dioxide|
+----+----+---------------+-----------------------+-------+---------------+----------------+
+----+----+---------------+-----------------------+-------+---------------+----------------+

+----------+--------+-----------------+-----------------------+-------+---------------+----------------+
|date      |time    |carbon_monoxide  |non_methane_hydrocarbon|benzene|nitrogen_oxides|nitrogen_dioxide|
+----------+--------+-----------------+-----------------------+-------+---------------+----------------+
|12/03/2004|04.00.00|2.161363636363637|10.0                   |1.1    |21.0           |32.0            |
|12/03/2004|12.00.00|2.1              |114.0  

In [6]:
df_final = processor._enrich_data(df_step4)

print("Step 5 Complete: Enriched data with timestamp and time features.")
print("Final processed DataFrame:")
df_final.printSchema()
df_final.show(5, truncate=False)

- Step 5: Enriching with timestamp and time features...
Step 5 Complete: Enriched data with timestamp and time features.
Final processed DataFrame:
root
 |-- carbon_monoxide: double (nullable = false)
 |-- non_methane_hydrocarbon: double (nullable = false)
 |-- benzene: double (nullable = false)
 |-- nitrogen_oxides: double (nullable = false)
 |-- nitrogen_dioxide: double (nullable = false)
 |-- event_timestamp: timestamp (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- hour_of_day: integer (nullable = true)

+-----------------+-----------------------+-------+---------------+----------------+-------------------+----+-----+---+-----------+-----------+
|carbon_monoxide  |non_methane_hydrocarbon|benzene|nitrogen_oxides|nitrogen_dioxide|event_timestamp    |year|month|day|day_of_week|hour_of_day|
+-----------------+-----------------------+-------+---------------+-

In [7]:
spark_manager.stop_session()

Spark session stopped.
